In [ ]:
import gc
import os
import time

import numpy as np
import pandas as pd
import torch
import tsai.all as ts
from fastai.torch_core import set_seed
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Same seed as the main CV in 05d, reset before every LOIO fold
SEED = 42

# Leave-one-individual-out (LOIO) validation -- HydraMultiROCKET

Same LOIO design as `08a_individual_validation_features_ml.ipynb` (see that
notebook for the full diagnostics, threshold rationale, and per-class fold
coverage check) -- this notebook loads the per-burst fold lookups built there
and runs **HydraMultiROCKET** (from `05d_timeseries_classification_dl.ipynb`)
on the raw accelerometer time series, instead of the tabular feature set used
by RF/TabPFN.

Recap of scope:
- **Datasets**: Burst_1/uncorrected (raw-data baseline) and Burst_2/uncorrected
  (the final classifier's burst length).
  Bursts 2-4 are the same recordings as Burst_1 at finer child-window
  granularities, so each is run and reported separately, never pooled
  together.
- **Target**: behaviour only (the activity target is not used).
- An individual is a held-out test fold in a given burst only if they have
  >=30 total samples *in that burst*; smaller individuals stay in every
  fold's training set but are never the test fold. This is applied
  independently per burst -- Burst_2's eligible set (17/20) is larger than
  Burst_1's (13/20) since it has ~2x more rows per individual.
- **The behaviour model trains on the full label space -- no classes are
  dropped from training.** It sees all 8 behaviour classes, matching the
  main CV models in `05d_timeseries_classification_dl.ipynb`. Running,
  Self-scratching, and Sleeping are excluded only when computing pooled
  metrics (see the pooling cell), not from training -- see 08a for the full
  rationale.

Class imbalance handling mirrors `05d_timeseries_classification_dl.ipynb`:
`use_class_weight` weights the training loss by inverse class frequency via
`dls.train.cws`, recomputed fresh from each fold's own training split.

## Load the per-burst fold lookups built in 08a

In [ ]:
BURSTS = ["burst_1", "burst_2"]
# Classes excluded from *pooled metrics only* (not from training -- see
# process_parquet_file and 08a's introduction for the full rationale).
DROPPED_MULTICLASS_CLASSES = ["Running", "Self-scratching", "Sleeping"]
TRAIN_ONLY_SENTINEL = "train_only"

fold_lookups = {}
per_burst_eligibility = {}
for burst in BURSTS:
    path = f"../data/temp/indices_loio_fold_lookup_{burst}.csv"
    fold_lookup = pd.read_csv(path)
    fold_lookups[burst] = fold_lookup

    eligible = sorted(
        fold_lookup.loc[fold_lookup["fold"] != TRAIN_ONLY_SENTINEL, "fold"]
        .unique()
        .tolist()
    )
    per_burst_eligibility[burst] = {"eligible": eligible}
    print(f"{burst}: eligible LOIO fold individuals ({len(eligible)}): {eligible}")

# The set of classes pooled multiclass metrics are restricted to -- the
# complement of DROPPED_MULTICLASS_CLASSES within whatever classes actually
# appear (unioned across bursts).
all_multiclass_classes = sorted(
    set().union(*[fold_lookups[b]["behaviour"].unique() for b in BURSTS])
)
RETAINED_MULTICLASS_CLASSES = [
    c for c in all_multiclass_classes if c not in DROPPED_MULTICLASS_CLASSES
]
print(
    f"Retained multiclass classes for metrics ({len(RETAINED_MULTICLASS_CLASSES)}): "
    f"{RETAINED_MULTICLASS_CLASSES}"
)

## Build the time-series X array once, aligned to the LOIO fold lookup

In [ ]:
def build_ts_arrays(acc, fold_lookup):
    """Build the time-series X array once per file, aligned to LOIO fold assignments.

    Mirrors build_ts_arrays in 05d_timeseries_classification_dl.ipynb, but
    merges directly against the precomputed LOIO fold_lookup (which already
    has burst filtered to the target burst and fold set to the held-out
    individual's name, or the train_only sentinel) instead of filtering
    fold_lookup by burst internally.

    Args:
        acc (pd.DataFrame): Raw accelerometer long-format data straight from
            the parquet file (columns: Ind_ID, new_burst, feature, plus
            numeric feature/time columns -- no "id" column of its own).
        fold_lookup (pd.DataFrame): LOIO fold lookup with columns Ind_ID,
            new_burst, behaviour, activity, fold

    Returns:
        tuple: (X, merged) where X is an (n_samples, 3, n_features) array and
            merged is a DataFrame with columns id, Ind_ID, new_burst, idx,
            fold, activity, behaviour -- merged.loc[i, "idx"] indexes X's
            first axis for sample i.
    """
    m = fold_lookup[["Ind_ID", "new_burst", "behaviour", "activity", "fold"]]

    acc = acc.copy()
    acc["new_burst"] = acc["new_burst"].astype(np.int64)
    acc["id"] = acc["Ind_ID"].astype(str) + "_" + acc["new_burst"].astype(str)
    id_mapping = (
        acc[["id", "Ind_ID", "new_burst"]].drop_duplicates().reset_index(drop=True)
    )

    merged = id_mapping.merge(m, on=["Ind_ID", "new_burst"], how="inner")
    merged = merged.reset_index(drop=True)
    merged["idx"] = merged.index

    feature_cols = [col for col in acc.columns if col.isdigit()]
    n_features = len(feature_cols)
    n_samples = len(merged)
    print(f"Number of samples: {n_samples}, number of features: {n_features}")

    X = np.zeros((n_samples, 3, n_features), dtype=np.float64)
    feature_to_idx = {"X": 0, "Y": 1, "Z": 2}

    for i, id_val in enumerate(merged["id"]):
        id_data = acc[acc["id"] == id_val]
        for feature_type, axis_idx in feature_to_idx.items():
            feature_data = id_data[id_data["feature"] == feature_type]
            if len(feature_data) > 0:
                X[i, axis_idx, :] = feature_data[feature_cols].values[0]
            else:
                print(f"Warning: No data found for ID {id_val}, feature {feature_type}")

    return X.astype(np.float64), merged


def create_dsets_for_individual(X, merged, held_out):
    """Create a behaviour TSDatasets object for one LOIO fold from prebuilt X/labels.

    No classes are dropped here -- the model trains on the full label space
    (see the notebook introduction and process_parquet_file). Sparse classes
    are excluded only when computing pooled metrics afterward, in the
    pooling cell.

    Args:
        X (np.ndarray): Prebuilt (n_samples, 3, n_features) array
        merged (pd.DataFrame): Output of build_ts_arrays, with idx/fold/
            behaviour columns
        held_out (str): Held-out individual name; train = fold != this value,
            valid = fold == this value

    Returns:
        TSDatasets: Dataset object ready for use with tsai DataLoaders
    """
    train_indices = merged.loc[merged["fold"] != held_out, "idx"].tolist()
    valid_indices = merged.loc[merged["fold"] == held_out, "idx"].tolist()
    splits = (train_indices, valid_indices)

    y = merged["behaviour"].to_numpy(dtype="<U20")

    tfms = [None, [ts.Categorize()]]
    dsets = ts.TSDatasets(
        X.astype(np.float64), y=y, tfms=tfms, splits=splits, inplace=True
    )
    return dsets

## HydraMultiROCKET LOIO training/evaluation function

In [ ]:
def run_hydra_multirocket(dset, use_class_weight=True, seed=SEED):
    """Train and evaluate HydraMultiROCKET on one LOIO fold.

    Uses the same seeding, dataloaders, loss weighting and training schedule
    as run_models_on_dataset in 05d_timeseries_classification_dl.ipynb,
    trimmed to HydraMultiROCKET only. As in 05d, the seed is reset immediately
    before building the dataloaders and the model, so each fold starts from
    the same RNG state and the run is reproducible. Unlike 05d, the per-fold
    metrics here come from fastai's live metrics (learn.recorder).

    Predictions/targets/decoded-labels are pulled once via learn.get_preds
    (validation set, ds_idx=1 by default) instead of going through
    ClassificationInterpretation, which doesn't store preds/targs as
    attributes (they're fetched on-demand inside methods like
    confusion_matrix()). Getting them directly also lets the confusion
    matrix be built with sklearn's confusion_matrix, matching
    features_ml_models/features_tabpfn_model in 08a/08b, and avoids a second
    redundant inference pass.

    roc_auc is computed post-hoc from these predictions, guarded by
    try/except -- mirroring 08a/08b. It is deliberately NOT a live fastai
    Learner metric: a held-out individual's fold can easily be missing one
    or more of the 8 behaviour classes entirely (most individuals have no
    Running, Self-scratching or Sleeping windows), which makes sklearn's
    multiclass ROC-AUC undefined for that fold. As a live metric, that
    raises inside fastai's Recorder at the after_validate event and aborts
    fit_one_cycle entirely -- losing accuracy, balanced_acc, and the
    confusion matrix along with it, not just roc_auc. Computing it afterward
    means a missing class only costs the roc_auc value (NaN), not the whole
    fold.
    """
    set_seed(seed, reproducible=True)
    dls = ts.TSDataLoaders.from_dsets(dset.train, dset.valid, bs=[64, 128])

    metrics = [
        ts.accuracy,
        ts.BalancedAccuracy(),
        ts.F1Score(average="weighted"),
        ts.Precision(average="weighted"),
        ts.Recall(average="weighted"),
        ts.F1Score(average="macro"),
        ts.Precision(average="macro"),
        ts.Recall(average="macro"),
    ]

    loss_func = (
        ts.CrossEntropyLossFlat(weight=dls.train.cws) if use_class_weight else None
    )

    results = []
    confusion_data = []
    model_name = "HydraMultiRocket"

    try:
        model = ts.create_model(ts.HydraMultiRocket, dls=dls)
        learn = ts.Learner(dls, model, metrics=metrics, loss_func=loss_func)

        start = time.time()
        lr_max = learn.lr_find()
        learn.fit_one_cycle(50, lr_max)
        elapsed = time.time() - start

        vals = learn.recorder.values[-1]

        classes = dset.vocab
        n_classes = len(classes)

        y_proba, y_true, y_pred = learn.get_preds(with_decoded=True)
        y_proba = y_proba.cpu().numpy()
        y_true = y_true.cpu().numpy()
        y_pred = y_pred.cpu().numpy()

        try:
            roc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted")
        except Exception:
            roc = np.nan

        results.append(
            {
                "model": model_name,
                "train_loss": vals[0],
                "valid_loss": vals[1],
                "accuracy": vals[2],
                "roc_auc": roc,
                "balanced_acc": vals[3],
                "f1_score": vals[4],
                "precision": vals[5],
                "recall": vals[6],
                "f1_macro": vals[7],
                "precision_macro": vals[8],
                "recall_macro": vals[9],
                "time": int(elapsed),
            }
        )

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
        for actual_idx, actual in enumerate(classes):
            for pred_idx, predicted in enumerate(classes):
                confusion_data.append(
                    {
                        "model": model_name,
                        "actual_label": actual,
                        "predicted_label": predicted,
                        "count": cm[actual_idx, pred_idx],
                    }
                )
    except Exception as e:
        print(f"Error with {model_name}: {e!s}")

    return pd.DataFrame(results), pd.DataFrame(confusion_data)

## LOIO orchestration for the behaviour target

In [ ]:
def process_parquet_file(
    file_path, fold_lookup, eligible_individuals, use_class_weight=True
):
    """Run LOIO cross-validation for the behaviour target on one file.

    The model trains on the full label space -- no classes are dropped here.
    It sees all 8 behaviour classes, matching the main CV models in
    05d_timeseries_classification_dl.ipynb. Sparse classes are excluded only
    when computing pooled metrics afterward (see the pooling cell), not from
    training.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    acc = pd.read_parquet(file_path)
    X, merged = build_ts_arrays(acc, fold_lookup)

    metrics, confs = [], []
    for held_out in eligible_individuals:
        n_test = int((merged["fold"] == held_out).sum())
        if n_test == 0:
            continue

        print(f"  Running Behaviour classification, held-out individual {held_out}")
        dset = create_dsets_for_individual(X, merged, held_out)
        met, conf = run_hydra_multirocket(dset, use_class_weight=use_class_weight)
        if met.empty:
            # run_hydra_multirocket hit an exception internally (its own
            # try/except already printed the reason) and returned empty
            # frames -- skip this individual's row rather than appending
            # a columnless frame, which would otherwise make the
            # concat+column-select below raise a KeyError and lose every
            # OTHER individual's (successfully computed) results too.
            continue
        for d in (met, conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["held_out_individual"] = held_out
            d["n_test_samples"] = n_test
        metrics.append(met)
        confs.append(conf)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "held_out_individual",
        "n_test_samples",
        "model",
        "train_loss",
        "valid_loss",
        "accuracy",
        "roc_auc",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "held_out_individual",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: pd.concat(lst)[cols] if lst else pd.DataFrame(columns=cols)
    return {
        "multiclass_metrics": fin(metrics, metric_columns),
        "multiclass_conf": fin(confs, conf_columns),
    }

## Run LOIO across datasets

In [ ]:
DATASETS = [
    ("burst_1", "uncorrected"),
    ("burst_2", "uncorrected"),
]

all_results = []
for burst, correction_type in DATASETS:
    fold_lookup = fold_lookups[burst]
    eligible = per_burst_eligibility[burst]["eligible"]
    pq_file_path = (
        f"../data/temp/acc_tsai/{burst.capitalize()}/"
        f"annotated_acc_tsai_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file(pq_file_path, fold_lookup, eligible)
    all_results.append(result)

multiclass_metrics = pd.concat(
    [r["multiclass_metrics"] for r in all_results], ignore_index=True
)
multiclass_conf = pd.concat(
    [r["multiclass_conf"] for r in all_results], ignore_index=True
)

## Pool results and inspect per-individual breakdown

In [ ]:
def pooled_metrics_from_confusion(conf_df, model_name, retained_classes=None):
    """Reconstruct pooled y_true/y_pred from a summed confusion matrix and
    compute overall metrics on it -- more meaningful than averaging
    per-individual metrics when many individuals have zero test examples of
    some classes (e.g. grooming for individuals who never groom).

    Args:
        retained_classes (list, optional): If given, metrics are computed
            only over rows whose TRUE label is one of these classes, keeping
            whatever the model actually predicted (even a class outside this
            set counts as a genuine miss). Training itself is never
            restricted -- see process_parquet_file -- this only controls
            which true-label rows count toward the reported metrics, so the
            model stays an 8-class classifier and only the comparison metric
            is restricted to the well-populated classes.
    """
    sub = conf_df[conf_df["model"] == model_name]
    if sub.empty or sub["count"].sum() == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}
    y_true = np.repeat(sub["actual_label"].to_numpy(), sub["count"].to_numpy())
    y_pred = np.repeat(sub["predicted_label"].to_numpy(), sub["count"].to_numpy())

    if retained_classes is not None:
        mask = np.isin(y_true, retained_classes)
        y_true, y_pred = y_true[mask], y_pred[mask]
        labels_kwarg = {"labels": retained_classes}
    else:
        labels_kwarg = {}

    if len(y_true) == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}

    # balanced_accuracy_score has no `labels` param, so when restricting to a
    # subset of classes we use macro recall instead -- mathematically the
    # same thing balanced accuracy is (mean per-class recall), and this
    # matches the convention already used elsewhere (recall_macro ==
    # balanced_acc throughout 05a-05d's own metrics).
    balanced_acc = (
        recall_score(y_true, y_pred, average="macro", zero_division=0, **labels_kwarg)
        if retained_classes is not None
        else balanced_accuracy_score(y_true, y_pred)
    )

    return {
        "model": model_name,
        "n_pooled_test_samples": len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": balanced_acc,
        "f1_score": f1_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "recall": recall_score(
            y_true, y_pred, average="weighted", zero_division=0, **labels_kwarg
        ),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0, **labels_kwarg
        ),
    }


def pooled_summary_per_dataset(conf_df, model_name, retained_classes=None):
    """Pooled metrics computed separately per (burst, correction_type), so
    datasets are never mixed into a single pool (Burst_2 is the same
    recordings as Burst_1 at finer windowing -- pooling across bursts would
    leak near-duplicate windows across train/test)."""
    rows = []
    for (burst, correction_type), grp in conf_df.groupby(["burst", "correction_type"]):
        m = pooled_metrics_from_confusion(
            grp, model_name, retained_classes=retained_classes
        )
        m["burst"] = burst
        m["correction_type"] = correction_type
        rows.append(m)
    cols = ["burst", "correction_type"] + [
        c for c in rows[0] if c not in ("burst", "correction_type")
    ]
    return pd.DataFrame(rows)[cols]


pooled_multiclass_summary = pooled_summary_per_dataset(
    multiclass_conf, "HydraMultiRocket", retained_classes=RETAINED_MULTICLASS_CLASSES
)
print("Pooled multiclass LOIO metrics per dataset (restricted to retained classes):")
print(pooled_multiclass_summary)

# Per-individual breakdown, alongside their sample counts, for transparency --
# this is the honest complement to the pooled summary: it shows how much the
# pooled result is driven by well-sampled vs. sparse individuals. Note these
# per-individual accuracy/balanced_acc/f1_macro values come straight from
# run_hydra_multirocket and are NOT restricted to RETAINED_MULTICLASS_CLASSES
# (that restriction is only applied in the pooled summary above) -- they
# reflect genuine 8-class performance for that held-out individual.
per_individual_cols = [
    "burst",
    "correction_type",
    "held_out_individual",
    "n_test_samples",
    "model",
    "accuracy",
    "balanced_acc",
    "f1_macro",
]
per_individual_multiclass = multiclass_metrics[per_individual_cols].sort_values(
    ["burst", "correction_type", "n_test_samples"], ascending=[True, True, False]
)
print("\nPer-individual multiclass results (unrestricted, all 8 classes):")
print(per_individual_multiclass)

## Save outputs

In [ ]:
multiclass_metrics.to_csv(
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_loio_metrics.csv",
    index=False,
)
multiclass_conf.to_csv(
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_loio_confusion.csv",
    index=False,
)
pooled_multiclass_summary.to_csv(
    "../data/output/comparison_behaviour/behaviour_acceleration_dl_loio_pooled_summary.csv",
    index=False,
)